In [1]:
# ==========================================================
# Notebook 03: Query Expansion and Ranking
# ==========================================================

import numpy as np
import pandas as pd
import faiss
import pickle

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# Load FAISS index

index = faiss.read_index("indexes/faiss_index.bin")

# Load metadata

with open("indexes/metadata.pkl", "rb") as file:

    metadata = pickle.load(file)

print("FAISS index loaded successfully!")

FAISS index loaded successfully!


In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded.")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 707c0f1e-0601-4c95-bd4c-67eef05ca7e1)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model loaded.


In [4]:
def basic_search(query, model, index, metadata, top_k=5):

    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):

        doc = metadata[idx]

        results.append(
            {
                "title": doc["title"],
                "category": doc["category"],
                "distance": float(distance),
            }
        )

    return pd.DataFrame(results)

In [5]:
basic_search(query="How do I study AI?", model=model, index=index, metadata=metadata)

,title,category,distance
0,Artificial Intelligence for Beginners,AI,0.934182
1,Python Programming,Programming,0.962770
2,Machine Learning Basics,ML,1.202092
3,Natural Language Processing,NLP,1.302101
4,Deep Learning Introduction,DL,1.567046


In [6]:
synonym_dictionary = {
    "ai": ["artificial intelligence"],
    "ml": ["machine learning"],
    "nlp": ["natural language processing"],
    "dl": ["deep learning"],
    "cv": ["computer vision"],
}

In [7]:
def expand_query(query, synonym_dict):

    words = query.lower().split()

    expanded_words = []

    for word in words:

        expanded_words.append(word)

        if word in synonym_dict:

            expanded_words.extend(synonym_dict[word])

    expanded_query = " ".join(expanded_words)

    return expanded_query

In [8]:
query = "How do I study AI and NLP?"

expanded_query = expand_query(query, synonym_dictionary)

print("Original Query:\n", query)

print("\nExpanded Query:\n", expanded_query)

Original Query:
 How do I study AI and NLP?

Expanded Query:
 how do i study ai artificial intelligence and nlp?


In [9]:
def expanded_search(query, model, index, metadata, synonym_dict, top_k=5):

    expanded_query = expand_query(query, synonym_dict)

    query_embedding = model.encode([expanded_query]).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):

        doc = metadata[idx]

        results.append(
            {
                "title": doc["title"],
                "category": doc["category"],
                "distance": float(distance),
            }
        )

    return pd.DataFrame(results)

In [10]:
print("Basic Search:\n")

display(basic_search(query="Learn AI", model=model, index=index, metadata=metadata))

print("\nExpanded Search:\n")

display(
    expanded_search(
        query="Learn AI",
        model=model,
        index=index,
        metadata=metadata,
        synonym_dict=synonym_dictionary,
    )
)

Basic Search:



,title,category,distance
0,Artificial Intelligence for Beginners,AI,0.803823
1,Machine Learning Basics,ML,0.875150
2,Python Programming,Programming,0.911562
3,Natural Language Processing,NLP,1.144732
4,Deep Learning Introduction,DL,1.197167



Expanded Search:



,title,category,distance
0,Artificial Intelligence for Beginners,AI,0.669573
1,Machine Learning Basics,ML,0.827143
2,Python Programming,Programming,0.849588
3,Natural Language Processing,NLP,1.180130
4,Deep Learning Introduction,DL,1.205846


In [11]:
acronym_map = {
    "llm": "large language model",
    "rag": "retrieval augmented generation",
    "bert": "bidirectional encoder representations from transformers",
}

print(acronym_map)

{'llm': 'large language model', 'rag': 'retrieval augmented generation', 'bert': 'bidirectional encoder representations from transformers'}


In [12]:
def keyword_overlap(query, text):

    query_words = set(query.lower().split())

    text_words = set(text.lower().split())

    return len(query_words.intersection(text_words))

In [13]:
keyword_overlap("learn ai", "Artificial Intelligence for Beginners")

0

In [14]:
def rank_results(query, search_results):

    ranked = search_results.copy()

    ranked["keyword_score"] = ranked["title"].apply(lambda x: keyword_overlap(query, x))

    ranked["final_score"] = -ranked["distance"] + 0.2 * ranked["keyword_score"]

    ranked = ranked.sort_values(by="final_score", ascending=False)

    return ranked

In [15]:
retrieved = expanded_search(
    query="Learn AI",
    model=model,
    index=index,
    metadata=metadata,
    synonym_dict=synonym_dictionary,
)

ranked = rank_results(query="Learn AI", search_results=retrieved)

ranked

,title,category,distance,keyword_score,final_score
0,Artificial Intelligence for Beginners,AI,0.669573,0,-0.669573
1,Machine Learning Basics,ML,0.827143,0,-0.827143
2,Python Programming,Programming,0.849588,0,-0.849588
3,Natural Language Processing,NLP,1.180130,0,-1.180130
4,Deep Learning Introduction,DL,1.205846,0,-1.205846


In [16]:
def intelligent_search(query, model, index, metadata, synonym_dict, top_k=5):

    retrieved = expanded_search(
        query=query,
        model=model,
        index=index,
        metadata=metadata,
        synonym_dict=synonym_dict,
        top_k=top_k,
    )

    ranked = rank_results(query=query, search_results=retrieved)

    return ranked

In [17]:
intelligent_search(
    query="Learn AI",
    model=model,
    index=index,
    metadata=metadata,
    synonym_dict=synonym_dictionary,
)

,title,category,distance,keyword_score,final_score
0,Artificial Intelligence for Beginners,AI,0.669573,0,-0.669573
1,Machine Learning Basics,ML,0.827143,0,-0.827143
2,Python Programming,Programming,0.849588,0,-0.849588
3,Natural Language Processing,NLP,1.180130,0,-1.180130
4,Deep Learning Introduction,DL,1.205846,0,-1.205846


In [18]:
while True:

    query = input("\nEnter Query (or quit): ")

    if query.lower() == "quit":
        break

    expanded = expand_query(query, synonym_dictionary)

    print("\nExpanded Query:")

    print(expanded)

    print("\nRetrieved Results:")

    display(
        intelligent_search(
            query=query,
            model=model,
            index=index,
            metadata=metadata,
            synonym_dict=synonym_dictionary,
        )
    )


Expanded Query:
nlp natural language processing

Retrieved Results:


,title,category,distance,keyword_score,final_score
0,Natural Language Processing,NLP,0.835917,0,-0.835917
1,Machine Learning Basics,ML,1.584349,0,-1.584349
2,Python Programming,Programming,1.619626,0,-1.619626
3,Artificial Intelligence for Beginners,AI,1.669225,0,-1.669225
4,Deep Learning Introduction,DL,1.754993,0,-1.754993
